# 📘 Media‑to‑ELAN Automatic Segmentation Tool

This application provides an end‑to‑end workflow for automatically segmenting **audio or video files** and generating a structured **ELAN (.eaf)** annotation file.  
It is designed for linguistic fieldwork, corpus creation, and annotation bootstrapping, with a focus on transparency, reproducibility, and clean tier architecture.

---

## 🎯 Purpose

The tool automatically:

- extracts audio from video files  
- converts audio to a format compatible with `auditok`  
- detects speech regions based on energy thresholds  
- applies buffer margins and overlap resolution  
- generates a structured ELAN file with three tiers  
- links the original media file inside the EAF  

This enables rapid creation of annotation scaffolding for transcription and translation workflows.

---

## 🧩 Supported Input Formats

- **Audio:** WAV, MP3  
- **Video:** MP4, MOV, AVI, MKV  

Video files are automatically processed by extracting their audio track.

---

## 🔊 Audio Processing Pipeline

1. **Video audio extraction**  
   If the input is a video file, the audio track is extracted using `pydub`.

2. **Forced PCM conversion**  
   All audio is converted to **16‑bit PCM WAV**, ensuring compatibility with `auditok`, which only accepts sample widths of 1, 2, or 4 bytes.

3. **Speech region detection**  
   `auditok.split()` identifies speech segments using:
   - energy threshold  
   - maximum allowed silence inside a region  

4. **Buffering and extension**  
   Each detected region is expanded by a user‑defined buffer (ms) on both sides, unless adjacent segments are too close.

5. **Overlap resolution**  
   Adjacent buffered segments are adjusted to avoid overlap by splitting the midpoint.

6. **Minimum duration safeguard**  
   Segments shorter than the minimum duration are extended to meet the threshold.

---

## 🗂 Tier Architecture

The user provides a **Tier Suffix** (e.g., `ABC`) in the UI.

The application then generates three tiers:

| Tier Type | Tier Name Pattern | Description |
|----------|-------------------|-------------|
| Parent (time‑alignable) | `MJK@<suffix>` | Main segmentation tier |
| Child (Symbolic_Association) | `TPI@<suffix>` | Translation tier |
| Child (Symbolic_Association) | `ENG@<suffix>` | English translation tier |

Example with suffix `ABC`:

```
MJK@ABC
 ├── TPI@ABC
 └── ENG@ABC
```

All dependent tiers reference the parent tier’s annotations.

---

## 📝 Annotation Generation

For each detected segment:

- A parent annotation is created on `MJK@<suffix>`  
- Two dependent annotations are created on:
  - `TPI@<suffix>`
  - `ENG@<suffix>`

All dependent annotations are empty placeholders, ready for transcription and translation.

---

## 🎛 User Interface (Tkinter)

The GUI provides:

- **File selection** (audio or video)  
- **Buffer (ms)**  
- **Max Silence (s)**  
- **Minimum Duration (ms)**  
- **Tier Suffix** (used to generate tier names)  
- **Run** and **Reset Defaults** buttons  
- Status display showing number of generated annotations  

The interface is designed for clarity and ease of use in fieldwork or batch‑processing contexts.

---

## 📦 Output

The tool produces:

- A fully structured **ELAN (.eaf)** file  
- Linked to the original media file (with correct MIME type)  
- With all three tiers created and aligned  
- With all detected segments annotated  

The `.eaf` file is saved next to the input media file.

---

## 🧹 Temporary File Handling

The application automatically removes:

- temporary audio extracted from video  
- temporary PCM‑converted audio  

This ensures a clean working directory.

---

## 🚀 Summary

This application provides a robust, reproducible workflow for generating ELAN segmentation files from audio or video.  
It automates:

- media handling  
- segmentation  
- tier creation  
- annotation scaffolding  
- file linking  

…while giving the user full control over segmentation parameters and tier naming conventions.

It is ideal for linguists, annotators, and researchers who need fast, consistent segmentation across large datasets.

---

If you'd like, I can also generate:

- a GitHub‑ready README.md  
- a flowchart diagram in Markdown  
- a “How to Use” tutorial section  
- a troubleshooting section for common errors  

Just tell me what you want to add next.


In [2]:
!pip install auditok


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ----------------------------------- ---- 1.3/1.5 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.9 MB/s  0:00:00


In [3]:
import os
import tkinter as tk
from tkinter import filedialog, messagebox
import auditok
import pympi
from pydub import AudioSegment


def process_file(input_file, buffer_value, max_silence, min_dur, tier_suffix):
    # --- Step 1: Detect whether input is video ---
    ext = os.path.splitext(input_file)[1].lower()
    is_video = ext in [".mp4", ".mov", ".avi", ".mkv"]

    # --- Step 2: Extract audio if needed ---
    if is_video:
        temp_audio = input_file + "_temp_audio.wav"
        AudioSegment.from_file(input_file).export(temp_audio, format="wav")
        audio_file = temp_audio
    else:
        audio_file = input_file

    # --- Step 3: Force-convert to 16-bit PCM WAV (auditok requirement) ---
    converted_audio = input_file + "_pcm.wav"
    AudioSegment.from_file(audio_file).set_sample_width(2).export(converted_audio, format="wav")
    audio_file = converted_audio

    # --- Step 4: Detect speech regions ---
    speech_regions = auditok.split(
        audio_file,
        energy_threshold=55,
        max_silence=max_silence
    )

    regions = sorted([(int(r.start * 1000), int(r.end * 1000)) for r in speech_regions])

    audio = AudioSegment.from_file(audio_file)
    audio_len_ms = len(audio)

    # --- Step 5: Create dynamic tier names ---
    default_tier = f"MJK@{tier_suffix}"
    translation_tier = f"TPI@{tier_suffix}"
    eng_tier = f"ENG@{tier_suffix}"

    # --- Step 6: Create EAF ---
    eaf = pympi.Elan.Eaf(file_path=None)
    
    # Remove auto-created default tier if present
    if "default" in eaf.get_tier_names():
        eaf.remove_tier("default")


    eaf.add_linguistic_type("ts")
    eaf.add_linguistic_type("tl", constraints="Symbolic_Association", timealignable=False)

    # Add tiers
    eaf.add_tier(default_tier, "ts")
    eaf.add_tier(translation_tier, "tl", parent=default_tier)
    eaf.add_tier(eng_tier, "tl", parent=default_tier)

    BUFFER = buffer_value
    MIN_DUR = min_dur

    # --- Step 7: Extend regions with buffer ---
    extended = []
    for i, (start_ms, end_ms) in enumerate(regions):
        left_gap = float("inf") if i == 0 else start_ms - regions[i - 1][1]
        right_gap = float("inf") if i == len(regions) - 1 else regions[i + 1][0] - end_ms

        new_start = start_ms - BUFFER if left_gap > BUFFER else start_ms
        new_end = end_ms + BUFFER if right_gap > BUFFER else end_ms

        new_start = max(0, new_start)
        new_end = min(audio_len_ms, new_end)

        if new_end <= new_start:
            new_end = min(audio_len_ms, new_start + MIN_DUR)

        extended.append([new_start, new_end])

    # --- Step 8: Resolve overlaps ---
    i = 0
    while i < len(extended) - 1:
        a_start, a_end = extended[i]
        b_start, b_end = extended[i + 1]

        if a_end > b_start:
            mid = (a_end + b_start) // 2
            new_a_end = max(a_start, mid)
            new_b_start = min(b_end, max(new_a_end, mid))

            if new_b_start >= b_end:
                b_end = min(audio_len_ms, new_b_start + MIN_DUR)

            extended[i] = [a_start, new_a_end]
            extended[i + 1] = [new_b_start, b_end]

            if i > 0:
                i -= 1
                continue
        i += 1

    # --- Step 9: Add annotations ---
    for start_ms, end_ms in extended:
        eaf.add_annotation(default_tier, start=start_ms, end=end_ms, value="")
        eaf.add_ref_annotation(translation_tier, default_tier, value="", time=start_ms)
        eaf.add_ref_annotation(eng_tier, default_tier, value="", time=start_ms)

    # --- Step 10: Link the media file ---
    mime_map = {
        ".mp4": "video/mp4",
        ".mov": "video/quicktime",
        ".avi": "video/x-msvideo",
        ".mkv": "video/x-matroska",
        ".wav": "audio/wav",
        ".mp3": "audio/mpeg"
    }
    mimetype = mime_map.get(ext, "application/octet-stream")

    eaf.add_linked_file(os.path.basename(input_file), mimetype=mimetype)

    # --- Step 11: Save EAF ---
    output_file = os.path.splitext(input_file)[0] + ".eaf"
    eaf.to_file(output_file)

    # --- Step 12: Cleanup ---
    if is_video and os.path.exists(temp_audio):
        os.remove(temp_audio)

    if os.path.exists(converted_audio):
        os.remove(converted_audio)

    # --- Step 13: UI feedback ---
    num_annotations = len(extended)
    messagebox.showinfo("Done", f"Annotations written to {output_file}\nNumber of annotations: {num_annotations}")
    annotation_label.config(text=f"ELAN file saved. Number of annotations: {num_annotations}")
    
    



def select_file():
    filename = filedialog.askopenfilename(
        title="Select Video or Audio File",
        filetypes=[
            ("Media files", "*.mp4 *.mov *.avi *.mkv *.wav *.mp3"),
            ("All files", "*.*")
        ]
    )
    if filename:
        selected_file.set(filename)
        file_label.config(text=f"Selected File: {os.path.basename(filename)}")


def run_processing():
    input_file = selected_file.get()
    if not input_file:
        messagebox.showerror("Error", "Please select a media file.")
        return
    try:
        buffer_value = int(buffer_var.get())
        max_silence = float(max_silence_var.get())
        min_dur = int(min_dur_var.get())
        tier_suffix = tier_suffix_var.get().strip()
    except ValueError:
        messagebox.showerror("Error", "Buffer/MinDur must be integers, Max Silence must be a number.")
        return

    if not tier_suffix:
        messagebox.showerror("Error", "Please enter a tier suffix.")
        return

    process_file(input_file, buffer_value, max_silence, min_dur, tier_suffix)


def reset_defaults():
    buffer_var.set("000")
    max_silence_var.set("0.3")
    min_dur_var.set("200")
    annotation_label.config(text="")


# --- GUI setup ---
root = tk.Tk()
root.title("Media to EAF Processor")
root.geometry("750x480")

selected_file = tk.StringVar()
buffer_var = tk.StringVar(value="000")
max_silence_var = tk.StringVar(value="0.3")
min_dur_var = tk.StringVar(value="200")
tier_suffix_var = tk.StringVar()   # <-- NEW

title_label = tk.Label(root, text="Segmentation of media files (audio/video → ELAN)", 
                       font=("Arial", 14, "bold"))
title_label.grid(row=0, column=0, columnspan=4, pady=10)

# File selection
tk.Button(root, text="Select File", command=select_file, width=15).grid(
    row=1, column=0, padx=10, pady=5)
file_label = tk.Label(root, text="No file selected", anchor="w")
file_label.grid(row=2, column=0, columnspan=4, sticky="w", padx=10)

# Buffer
tk.Label(root, text="Buffer (ms):").grid(row=3, column=0, sticky="w", padx=10, pady=5)
tk.Entry(root, textvariable=buffer_var, width=10).grid(
    row=3, column=1, sticky="w", padx=10, pady=5)
tk.Label(root, text="Extra margin added before/after regions").grid(
    row=3, column=2, sticky="w")

# Max Silence
tk.Label(root, text="Max Silence (s):").grid(row=4, column=0, sticky="w", padx=10, pady=5)
tk.Entry(root, textvariable=max_silence_var, width=10).grid(
    row=4, column=1, sticky="w", padx=10, pady=5)
tk.Label(root, text="Silence allowed inside a region").grid(
    row=4, column=2, sticky="w")

# Min Duration
tk.Label(root, text="Min Duration (ms):").grid(row=5, column=0, sticky="w", padx=10, pady=5)
tk.Entry(root, textvariable=min_dur_var, width=10).grid(
    row=5, column=1, sticky="w", padx=10, pady=5)
tk.Label(root, text="Safeguard: shortest allowed annotation").grid(
    row=5, column=2, sticky="w")

# NEW: Tier Suffix
tk.Label(root, text="Tier Suffix:").grid(row=6, column=0, sticky="w", padx=10, pady=5)
tk.Entry(root, textvariable=tier_suffix_var, width=15).grid(
    row=6, column=1, sticky="w", padx=10, pady=5)
tk.Label(root, text="Used to generate tier names (MJK@X, TPI@X, ENG@X)").grid(
    row=6, column=2, sticky="w")

# Run + Reset
tk.Button(root, text="Run", command=run_processing, width=10).grid(
    row=7, column=0, padx=10, pady=15)
tk.Button(root, text="Reset Defaults", command=reset_defaults, width=15).grid(
    row=7, column=1, padx=10, pady=15)

# Annotation count
annotation_label = tk.Label(root, text="", anchor="w", fg="blue")
annotation_label.grid(row=8, column=0, columnspan=4, sticky="w", padx=10, pady=5)

root.mainloop()

C:\Users\u1018237\AppData\Local\anaconda3\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
Matplotlib is building the font cache; this may take a moment.
C:\Users\u1018237\AppData\Local\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\u1018237\AppData\Local\anaconda3\Lib\tkinter\__init__.py", line 2074, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\u1018237\AppData\Local\Temp\ipykernel_22992\2620571326.py", line 172, in run_processing
    process_file(input_file, buffer_value, max_silence, min_dur, tier_suff